# {2..6}Mix prep - auxiliary notebook (internet ON, CPU is fine)

Stages everything the offline training notebook needs into `/kaggle/working/`:

1. pip deps (installed with `--target`, importable via `sys.path`)
2. the SR_CorrNet_SS repo (cloned from GitHub)
3. the pretrained 2-5-speaker checkpoint from HuggingFace
4. **{2..6}Mix**: N-speaker mixtures (2mix..6mix, train/valid/test)

**Corpus** is selected by `CORPUS` in the params cell:

- `wsj0` (default) - matches the corpus the pretrained checkpoint was trained on.
  Attach a dataset holding `csr_1_LDC93S6A.tgz` (e.g. `stsword/wsj0original`, 9.01 GB).
  The tarball is the raw LDC distribution: `.wv1` files are shorten-compressed NIST
  SPHERE, which only `sph2pipe` decodes, so this notebook builds it from source.
  Split follows the canonical WSJ0-mix recipe: tr/cv share `si_tr_s` speakers with
  disjoint utterances, tt comes from `si_dt_05` + `si_et_05`.
- `libri` - attach a LibriSpeech dataset with `train-clean-100` and `dev-clean`.
  tr / cv / tt are fully speaker-disjoint.

**After running:** Save Version so the output persists, then attach this notebook's
output to the training notebook.


In [ ]:
# ---- parameters -------------------------------------------------------------
import glob, os

CORPUS = "wsj0"           # "wsj0" | "libri"

# mixtures per speaker count, order matches COUNTS. Weighted toward 6mix.
COUNTS  = "2,3,4,5,6"
N_TRAIN = "2000,2000,3000,5000,8000"   # ~7.5 GB of wavs
N_VALID = "500,500,750,1250,2000"      # fixed-seed validation set (~1.9 GB)
N_TEST  = "500,500,500,500,500"        # fixed-seed test set (~1.4 GB)
SEED    = 1234

# where the 9 GB tarball is unpacked. /kaggle/temp, NOT /kaggle/working: working has a
# 20 GB quota that the generated mixtures (~11 GB) already most of, and the raw .wv1
# tree is another ~9 GB that we do not want persisted into the saved version.
WSJ0_RAW = "/kaggle/temp/wsj0_raw"
OUT_WAV  = "/kaggle/working/mix/wav"
OUT_SCP  = "/kaggle/working/mix/scp"

if CORPUS == "wsj0":
    WSJ0_TGZ = (glob.glob("/kaggle/input/*/csr_1_LDC93S6A.tgz")
                or glob.glob("/kaggle/input/*/*.tgz"))
    assert WSJ0_TGZ, "attach a dataset containing csr_1_LDC93S6A.tgz (e.g. stsword/wsj0original)"
    WSJ0_TGZ = WSJ0_TGZ[0]
    print("wsj0 tarball:", WSJ0_TGZ, f"({os.path.getsize(WSJ0_TGZ)/2**30:.2f} GB)")
else:
    TRAIN_ROOTS = (glob.glob("/kaggle/input/*/LibriSpeech/train-clean-100")
                   or glob.glob("/kaggle/input/*/train-clean-100"))
    EVAL_ROOTS  = (glob.glob("/kaggle/input/*/LibriSpeech/dev-clean")
                   or glob.glob("/kaggle/input/*/dev-clean"))
    assert TRAIN_ROOTS and EVAL_ROOTS, "attach a LibriSpeech dataset"
    print("train roots:", TRAIN_ROOTS, "\neval roots: ", EVAL_ROOTS)


In [ ]:
# ---- 1. pip deps into /kaggle/working/deps -----------------------------------
# Everything engine.py imports that the offline GPU image may not ship.
# matplotlib must be <3.10: the repo's util_writer uses the removed tostring_rgb().
!pip install --quiet --target=/kaggle/working/deps \
    loguru rotary-embedding-torch mir_eval tensorboardX torchinfo thop ptflops \
    "matplotlib<3.10" soundfile librosa pyyaml tqdm scipy huggingface_hub
print("deps staged")


In [ ]:
# ---- 2. repo + 3. pretrained checkpoint ---------------------------------------
!git clone --depth 1 https://github.com/dmlguq456/SR_CorrNet_SS.git /kaggle/working/SR_CorrNet_SS
from huggingface_hub import hf_hub_download
for f in ["model.pt", "config.yaml"]:
    hf_hub_download("shinuh/sr-corrnet-ss-1ch-wsj-var-2-5spk", f, local_dir="/kaggle/working/ckpt")
print("checkpoint staged")


In [ ]:
# ---- 3b. sph2pipe + unpack WSJ0 (skipped for CORPUS="libri") -------------------
# WSJ0 .wv1 are NIST SPHERE with shorten compression: soundfile/librosa cannot read
# them, sph2pipe is the reference decoder and uncompresses shorten transparently.
SPH2PIPE = "sph2pipe"
if CORPUS == "wsj0":
    !git clone --depth 1 https://github.com/robd003/sph2pipe.git /kaggle/working/sph2pipe_src
    !cd /kaggle/working/sph2pipe_src && gcc -o /kaggle/working/sph2pipe *.c -lm 2>/dev/null \
        || (cd /kaggle/working/sph2pipe_src && cmake . >/dev/null && make >/dev/null \
            && cp sph2pipe /kaggle/working/sph2pipe)
    SPH2PIPE = "/kaggle/working/sph2pipe"
    assert os.path.exists(SPH2PIPE), "sph2pipe build failed"
    !{SPH2PIPE} -h 2>&1 | head -3

    os.makedirs(WSJ0_RAW, exist_ok=True)
    # ~9 GB: a few minutes. -k so a re-run of this cell does not redo the whole thing.
    !tar xzkf {WSJ0_TGZ} -C {WSJ0_RAW} 2>/dev/null || true
    !du -sh {WSJ0_RAW}
    print("sph2pipe:", SPH2PIPE)


In [ ]:
# ---- 3c. locate the WSJ0 subsets ----------------------------------------------
# The internal layout of the LDC tarball varies (csr_1/11-N.1/wsj0/si_tr_s/... on the
# standard pressing), so discover the subset dirs instead of hardcoding disk numbers.
# Case-insensitive: some pressings ship uppercase directory names.
if CORPUS == "wsj0":
    from pathlib import Path

    def find_subsets(root, name):
        hits = [p for p in Path(root).rglob("*") if p.is_dir() and p.name.lower() == name]
        return sorted(str(p) for p in hits)

    TRAIN_ROOTS = find_subsets(WSJ0_RAW, "si_tr_s")
    EVAL_ROOTS  = find_subsets(WSJ0_RAW, "si_dt_05") + find_subsets(WSJ0_RAW, "si_et_05")
    print("si_tr_s :", TRAIN_ROOTS)
    print("si_dt/et:", EVAL_ROOTS)
    assert TRAIN_ROOTS, f"no si_tr_s under {WSJ0_RAW} - inspect the extracted tree"
    assert EVAL_ROOTS,  f"no si_dt_05/si_et_05 under {WSJ0_RAW} - inspect the extracted tree"
    n_wv1 = sum(len(list(Path(r).rglob("*.wv1"))) for r in TRAIN_ROOTS + EVAL_ROOTS)
    print("total .wv1 utterances:", n_wv1)


In [ ]:
# ---- 4a. mixture generator (verified locally) ---------------------------------
gen_src = '"""Generate {2..6}Mix: N-speaker mixtures from LibriSpeech or WSJ0, for SR-CorrNet fine-tuning.\n\nLayout (matches the repo\'s SCP convention, scp_dir/<n>mix/{tr,cv,tt}_{mix,sK}.scp):\n    out_dir/<n>mix/{tr,cv,tt}/{mix,s1..sN}/<key>.wav      8 kHz, 16-bit PCM, 4 s\n    scp_dir/<n>mix/{tr,cv,tt}_mix.scp                      lines: "<key> <abs path>"\n    scp_dir/<n>mix/{tr,cv,tt}_s<K>.scp\n\nConstruction per mixture: N distinct speakers, one random utterance each, resampled\nto 8k, random 4s crop (zero-padded if shorter), per-source gain U[-5,+5] dB,\nmixture = sum(sources); if peak > 0.9 the mixture AND all sources are rescaled by the\nsame factor (never independently -- that would corrupt SI-SNR targets).\n\nTwo corpora, selected with --corpus:\n\n  libri  LibriSpeech .flac, speaker id from the path (<root>/<spk>/<chapter>/x.flac).\n         Default split: train speakers from --train_roots; --eval_roots speakers are\n         halved into cv and tt, so all three partitions are speaker-disjoint.\n\n  wsj0   WSJ0 .wv1 (NIST SPHERE, shorten-compressed -- decoded via sph2pipe, which is\n         the only thing that reads them). Speaker id from the path (<root>/<spk>/x.wv1).\n         Canonical WSJ0-mix split (--valid_source train_utterances): tr and cv share\n         si_tr_s speakers but draw from disjoint utterance pools, and tt comes from\n         si_dt_05 + si_et_05 pooled. This is what WSJ0-2mix/3mix do, and what the\n         pretrained var-2-5spk checkpoint was trained under.\n\n.wv2 files are the second (secondary-mic) channel of the same recording and are\nignored -- WSJ0-mix uses .wv1 only.\n"""\nimport argparse\nimport io\nimport random\nimport re\nimport shutil\nimport subprocess\nfrom pathlib import Path\n\nimport numpy as np\nimport librosa\nimport soundfile as sf\n\nSR = 8000\nMAX_LEN = 32000  # 4 s @ 8 kHz\nMIN_DUR = MAX_LEN / SR  # 4 s: shorter utterances are dropped (see index_speakers)\n\n# Fraction of each train speaker\'s utterances held out for cv when the corpus draws\n# validation from the training speakers (canonical WSJ0-mix).\nCV_UTT_FRACTION = 0.1\n\n\n# ---------------------------------------------------------------- corpus readers\n\ndef sphere_header(path):\n    """{field: value} from a NIST SPHERE header, without decoding the audio.\n\n    Layout: b"NIST_1A\\n", then the header size in bytes as ASCII, then\n    "<name> -i <int>" / "-s<len> <str>" lines until "end_head". Parsing this is what\n    lets the >=4 s filter run over the whole corpus without paying for a shorten\n    decode of every file.\n    """\n    with open(path, "rb") as f:\n        magic = f.readline().strip()\n        if magic != b"NIST_1A":\n            raise ValueError(f"{path}: not a SPHERE file (magic={magic!r})")\n        size = int(f.readline().strip())\n        head = f.read(size - f.tell()).decode("ascii", "replace")\n    fields = {}\n    for m in re.finditer(r"^(\\S+)\\s+-(?:i|r)\\s+(\\S+)$", head, re.M):\n        fields[m.group(1)] = float(m.group(2))\n    return fields\n\n\ndef sphere_duration(path):\n    h = sphere_header(path)\n    return h["sample_count"] / h["sample_rate"]\n\n\ndef flac_duration(path):\n    info = sf.info(str(path))\n    return info.frames / info.samplerate\n\n\ndef load_sphere(path, sph2pipe):\n    """Decode a shorten-compressed .wv1 to a float array at SR via sph2pipe -> stdout."""\n    out = subprocess.run([sph2pipe, "-f", "wav", str(path)],\n                         stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)\n    wav, sr = sf.read(io.BytesIO(out.stdout), dtype="float32")\n    if wav.ndim > 1:\n        wav = wav[:, 0]\n    return librosa.resample(wav, orig_sr=sr, target_sr=SR) if sr != SR else wav\n\n\ndef load_flac(path, _sph2pipe):\n    wav, _ = librosa.load(str(path), sr=SR)\n    return wav\n\n\nCORPORA = {\n    "libri": {\n        "ext": "*.flac",\n        "spk_of": lambda p: p.parts[-3],\n        "duration": flac_duration,\n        "load": load_flac,\n        "needs_sph2pipe": False,\n    },\n    "wsj0": {\n        "ext": "*.wv1",\n        "spk_of": lambda p: p.parts[-2],\n        "duration": sphere_duration,\n        "load": load_sphere,\n        "needs_sph2pipe": True,\n    },\n}\n\n\n# ---------------------------------------------------------------- indexing\n\ndef index_speakers(roots, corpus):\n    """{speaker_id: [utterance paths]} across one or more corpus subset roots.\n\n    Only utterances >= 4 s are kept: a silent (all-zero) source segment gives the\n    SI-SNR loss a NaN gradient (torch.norm at 0), which destroys training.\n    """\n    c = CORPORA[corpus]\n    spk = {}\n    for root in roots:\n        for f in Path(root).rglob(c["ext"]):\n            try:\n                if c["duration"](f) >= MIN_DUR:\n                    spk.setdefault(c["spk_of"](f), []).append(f)\n            except (ValueError, KeyError, RuntimeError) as e:\n                print(f"  skipping unreadable {f}: {e}")\n    return {k: sorted(v) for k, v in spk.items() if len(v) >= 2}\n\n\ndef split_utterances(speakers, fraction, seed):\n    """Partition each speaker\'s utterance list into (majority, holdout).\n\n    Speakers are shared across the two halves; utterances are not. Every speaker\n    contributes at least one utterance to each half, so speakers with a single\n    usable utterance are dropped (index_speakers already requires >= 2).\n    """\n    keep, held = {}, {}\n    for sid, utts in speakers.items():\n        u = list(utts)\n        random.Random(f"{seed}:{sid}").shuffle(u)\n        n_held = max(1, int(round(len(u) * fraction)))\n        n_held = min(n_held, len(u) - 1)  # always leave >=1 for the majority half\n        held[sid], keep[sid] = u[:n_held], u[n_held:]\n    return keep, held\n\n\n# ---------------------------------------------------------------- generation\n\ndef load_clip(path, rng, corpus, sph2pipe):\n    wav = CORPORA[corpus]["load"](path, sph2pipe)\n    if len(wav) > MAX_LEN:\n        off = rng.integers(0, len(wav) - MAX_LEN)\n        wav = wav[off:off + MAX_LEN]\n    else:\n        wav = np.pad(wav, (0, MAX_LEN - len(wav)))\n    return wav\n\n\ndef make_partition(speakers, n_spks, n_mix, out_wav, out_scp, part, rng, corpus, sph2pipe):\n    spk_ids = sorted(speakers)\n    if len(spk_ids) < n_spks:\n        raise SystemExit(\n            f"{part}/{n_spks}mix: need {n_spks} distinct speakers, have {len(spk_ids)}")\n    dirs = ["mix"] + [f"s{k}" for k in range(1, n_spks + 1)]\n    for d in dirs:\n        (out_wav / d).mkdir(parents=True, exist_ok=True)\n    scp = {d: [] for d in dirs}\n    for i in range(n_mix):\n        chosen = rng.choice(spk_ids, size=n_spks, replace=False)\n        srcs = []\n        for sid in chosen:\n            utt = speakers[sid][rng.integers(len(speakers[sid]))]\n            gain = 10 ** (rng.uniform(-5, 5) / 20)\n            srcs.append(load_clip(utt, rng, corpus, sph2pipe) * gain)\n        mixture = np.sum(srcs, axis=0)\n        peak = np.abs(mixture).max()\n        if peak > 0.9:\n            srcs = [s * (0.9 / peak) for s in srcs]\n            mixture = mixture * (0.9 / peak)\n        key = f"{part}_{n_spks}mix_{i:06d}"\n        for d, wav in zip(dirs, [mixture] + srcs):\n            p = out_wav / d / f"{key}.wav"\n            sf.write(p, wav.astype(np.float32), SR, subtype="PCM_16")\n            scp[d].append(f"{key} {p.resolve()}\\n")\n    out_scp.mkdir(parents=True, exist_ok=True)\n    for d in dirs:\n        name = f"{part}_{\'mix\' if d == \'mix\' else d}.scp"\n        with open(out_scp / name, "w") as f:\n            f.writelines(scp[d])\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--corpus", choices=sorted(CORPORA), default="libri")\n    ap.add_argument("--train_roots", nargs="+", required=True,\n                    help="subset dirs holding training speakers "\n                         "(libri: train-clean-100; wsj0: si_tr_s)")\n    ap.add_argument("--eval_roots", nargs="+", required=True,\n                    help="subset dirs holding held-out speakers "\n                         "(libri: dev-clean; wsj0: si_dt_05 si_et_05)")\n    ap.add_argument("--valid_source", choices=["eval_speakers", "train_utterances"],\n                    default=None,\n                    help="where cv comes from. eval_speakers: halve --eval_roots "\n                         "speakers into cv/tt (fully speaker-disjoint; libri default). "\n                         "train_utterances: cv reuses train speakers via a disjoint "\n                         "utterance holdout and tt takes all of --eval_roots "\n                         "(canonical WSJ0-mix; wsj0 default).")\n    ap.add_argument("--sph2pipe", default="sph2pipe",\n                    help="sph2pipe binary, required for --corpus wsj0 (.wv1 is "\n                         "shorten-compressed SPHERE and nothing else decodes it)")\n    ap.add_argument("--out_wav", required=True)\n    ap.add_argument("--out_scp", required=True)\n    ap.add_argument("--counts", default="2,3,4,5,6")\n    ap.add_argument("--n_train", default="2000,2000,3000,5000,8000",\n                    help="mixtures per count (same order as --counts); weights training toward high N")\n    ap.add_argument("--n_valid", default="500,500,750,1250,2000")\n    ap.add_argument("--n_test", default="500,500,500,500,500")\n    ap.add_argument("--seed", type=int, default=1234)\n    args = ap.parse_args()\n\n    if args.valid_source is None:\n        args.valid_source = "train_utterances" if args.corpus == "wsj0" else "eval_speakers"\n\n    sph2pipe = args.sph2pipe\n    if CORPORA[args.corpus]["needs_sph2pipe"]:\n        resolved = shutil.which(sph2pipe) or (sph2pipe if Path(sph2pipe).is_file() else None)\n        if resolved is None:\n            raise SystemExit(\n                f"--corpus {args.corpus} needs sph2pipe to decode .wv1, but {sph2pipe!r} "\n                f"was not found. Build it from https://github.com/robd003/sph2pipe and "\n                f"pass --sph2pipe /path/to/sph2pipe.")\n        sph2pipe = resolved\n\n    counts = [int(c) for c in args.counts.split(",")]\n    n_tr = [int(c) for c in args.n_train.split(",")]\n    n_cv = [int(c) for c in args.n_valid.split(",")]\n    n_tt = [int(c) for c in args.n_test.split(",")]\n\n    print(f"indexing {args.corpus} (utterances >= {MIN_DUR:g}s only)...")\n    train_spk = index_speakers(args.train_roots, args.corpus)\n    eval_spk = index_speakers(args.eval_roots, args.corpus)\n    if not train_spk or not eval_spk:\n        raise SystemExit("no usable speakers found -- check --train_roots/--eval_roots")\n\n    if args.valid_source == "train_utterances":\n        # Canonical WSJ0-mix: cv shares tr\'s speakers but not its utterances; tt is\n        # the full held-out speaker set (si_dt_05 + si_et_05).\n        train_spk, cv_spk = split_utterances(train_spk, CV_UTT_FRACTION, args.seed)\n        tt_spk = eval_spk\n    else:\n        # valid and test must not share speakers: split the eval speaker set in half\n        eval_ids = sorted(eval_spk)\n        random.Random(args.seed).shuffle(eval_ids)\n        cv_spk = {k: eval_spk[k] for k in eval_ids[: len(eval_ids) // 2]}\n        tt_spk = {k: eval_spk[k] for k in eval_ids[len(eval_ids) // 2:]}\n\n    print(f"speakers: train={len(train_spk)} valid={len(cv_spk)} test={len(tt_spk)} "\n          f"(valid_source={args.valid_source})")\n\n    for n, a, b, c in zip(counts, n_tr, n_cv, n_tt):\n        sub = f"{n}mix"\n        for part, spk, cnt in [("tr", train_spk, a), ("cv", cv_spk, b), ("tt", tt_spk, c)]:\n            # fixed seed per (count, partition): valid/test are identical across runs\n            rng = np.random.default_rng(args.seed + 1000 * n + {"tr": 0, "cv": 1, "tt": 2}[part])\n            make_partition(spk, n, cnt, Path(args.out_wav) / sub / part,\n                           Path(args.out_scp) / sub, part, rng, args.corpus, sph2pipe)\n            print(f"{sub}/{part}: {cnt} mixtures done")\n\n\nif __name__ == "__main__":\n    main()\n'
open("/kaggle/working/generate_mix.py", "w").write(gen_src)
print("generator staged")


In [ ]:
# ---- 4b. generate {2..6}Mix (takes a while on CPU: ~1-2 h) ---------------------
import subprocess, sys
cmd = [sys.executable, "/kaggle/working/generate_mix.py",
       "--corpus", CORPUS,
       "--train_roots", *TRAIN_ROOTS, "--eval_roots", *EVAL_ROOTS,
       "--out_wav", OUT_WAV, "--out_scp", OUT_SCP,
       "--counts", COUNTS, "--n_train", N_TRAIN, "--n_valid", N_VALID, "--n_test", N_TEST,
       "--seed", str(SEED)]
if CORPUS == "wsj0":
    cmd += ["--sph2pipe", SPH2PIPE]
print(" ".join(cmd))
subprocess.run(cmd, check=True)
!du -sh /kaggle/working/mix /kaggle/working/deps /kaggle/working/SR_CorrNet_SS /kaggle/working/ckpt


Done. **Save Version** (so `/kaggle/working` persists), then attach this notebook's output to the training notebook as input data.
